# 03 · Picture the shortcut candidates (Kaggle, CPU only)

Draws the cases where the model flagged a disease with confidence but put almost no heat inside the
radiologist's box (`shortcut_candidates.csv`, from `scripts/analyse_results.py`). Each picture shows the
X-ray twice: plain with the box on the left, so chest drains, wires and text markers are easy to see,
and with the heatmap on the right. It also copies a few test X-rays for trying the demo.

**Setup** (step-by-step in the readme, section D)
1. *Add Input*: your `cxr-shortcut-check` dataset (code + trained model) and the NIH dataset `nih-chest-xrays/data`.
2. *Settings*: Accelerator **None**. No GPU or internet is needed, so this uses none of your GPU hours.
3. **Save Version → Save & Run All (Commit)**, then download `shortcut_cases.zip` from the **Output** tab.

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'   # the input folders are read-only
INPUT = Path('/kaggle/input')

def find_dir(marker, depth=4):
    """Folder under /kaggle/input that holds `marker`. Kaggle's mount paths vary, so search instead of hard-coding."""
    for d in range(1, depth + 1):
        hits = sorted(INPUT.glob('*/' * d + marker))
        if hits:
            return hits[0].parents[marker.count('/')]
    raise FileNotFoundError(f'Cannot find {marker} under {INPUT}. Add the missing input in the right-hand panel (Add Input).')

CODE = find_dir('scripts/render_cases.py')       # your cxr-shortcut-check dataset
DATA = str(find_dir('Data_Entry_2017.csv'))      # the NIH dataset (original 1024px images)
RUN = CODE / 'trained' / 'densenet121_wbce'
OUT = Path('/kaggle/working/shortcut_cases')
print('code:', CODE); print('data:', DATA)
print('model found:', (RUN / 'best_model.pt').exists(), '  case list found:', (RUN / 'shortcut_candidates.csv').exists())

In [ ]:
subprocess.run([sys.executable, 'scripts/render_cases.py',
                '--checkpoint', str(RUN / 'best_model.pt'), '--data-dir', DATA,
                '--cases', str(RUN / 'shortcut_candidates.csv'), '--out-dir', str(OUT),
                '--demo-samples', '2'], cwd=CODE, check=True)

In [ ]:
from IPython.display import Image as Show
display(Show(str(OUT / 'overview.png')))

In [ ]:
!cd /kaggle/working && zip -qr shortcut_cases.zip shortcut_cases && ls -lh shortcut_cases.zip